# Experiment: GraphTrust-SWE Gemma pilot on Kaggle

**This is a prepared pilot harness, not a submitted run or an empirical result.** The intended pilot is ten development issues × six conditions × two sampling seeds. It tests exact input budgets, parseability, memory/time behavior, and graph evidence use before freezing the holdout.
Response protocol v2 (`files_citations_v2`, record schema `2.0`) requires fresh contexts and 120 fresh attempts. Earlier outputs remain under their original contract and cannot satisfy this pilot.

Use a user-selected free T4 session with the local Gemma 4 E2B checkpoint attached through Kaggle's model interface. Google documents the supported inference path at [Gemma basic text inference](https://ai.google.dev/gemma/docs/capabilities/text/basic). Checkpoint bytes and the actual installed environment are recorded at runtime.

Default execution performs no GPU inference. Set `GRAPHTRUST_RUN_REAL=1` and supply the actual attachment paths listed below to execute the prepared pilot. The default bootstrap installs pinned Transformers/Jinja2 through internet access, preserving Kaggle's CUDA-compatible torch. A fully offline hashed-wheel bootstrap is optional. No training, paid API, publishing, competition submission, or GPU session launch is performed by this notebook.


In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

RUN_REAL = os.environ.get('GRAPHTRUST_RUN_REAL', '0') == '1'
BOOTSTRAP = os.environ.get('GRAPHTRUST_BOOTSTRAP', 'online')
MODEL_ID = 'google/gemma-4-E2B-it'
REQUIRED = ('GRAPHTRUST_PACKAGE_WHEEL', 'GRAPHTRUST_PACKAGE_SHA256', 'GRAPHTRUST_MODEL_DIR', 'GRAPHTRUST_CONTEXT_DIR', 'GRAPHTRUST_PROTOCOL_JSON')
OPTIONAL = ('GRAPHTRUST_WHEELHOUSE', 'GRAPHTRUST_ENV_LOCK', 'GRAPHTRUST_MODEL_MANIFEST')
config = {key: os.environ.get(key) for key in REQUIRED + OPTIONAL}
SEEDS = [17, 42]
MAX_CALLS = 120
working = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd() / 'runs' / 'kaggle-pilot'
print({'run_real': RUN_REAL, 'bootstrap': BOOTSTRAP, 'seeds': SEEDS, 'max_calls': MAX_CALLS, 'output_directory': str(working)})
if RUN_REAL:
    assert BOOTSTRAP in {'online', 'offline'}
    missing = [key for key in REQUIRED if not config[key]]
    if BOOTSTRAP == 'offline':
        missing += [key for key in ('GRAPHTRUST_WHEELHOUSE', 'GRAPHTRUST_ENV_LOCK') if not config[key]]
    assert not missing, 'Supply actual attachment paths and package hash: ' + ', '.join(missing)


## Attachments and bootstrap

Attach the prepared resource bundle (package wheel, its SHA-256, contexts/manifest, protocol JSON) and the chosen Google E2B model checkpoint. Supply their actual mounted paths through the environment variables; no model or dataset mount path is guessed. The context directory must contain ten qualified development issues with six exact-tokenizer arms each. **Do not attach reference patches, test patches, or evaluator gold.**

The default online bootstrap pins [Transformers 5.19.0](https://pypi.org/project/transformers/5.19.0/) and [Jinja2 3.1.6](https://pypi.org/project/Jinja2/3.1.6/); it installs [Accelerate 1.10.1](https://pypi.org/project/accelerate/1.10.1/) only if absent. Kaggle's preinstalled CUDA torch is retained and the full resolved environment is saved. Reproduction uses that recorded lock and checkpoint hash.

For an entirely offline run, set `GRAPHTRUST_BOOTSTRAP=offline` and attach `GRAPHTRUST_WHEELHOUSE` plus `GRAPHTRUST_ENV_LOCK`, a fully pinned requirements file with wheel hashes compatible with the selected runtime. `GRAPHTRUST_MODEL_MANIFEST` is optional precomputed provenance; if supplied, use a JSON `files` mapping of relative checkpoint paths to SHA-256. Runtime hashes are always recorded.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if RUN_REAL:
    wheel = Path(config['GRAPHTRUST_PACKAGE_WHEEL'])
    assert wheel.is_file() and sha256(wheel) == config['GRAPHTRUST_PACKAGE_SHA256'], 'Package checksum mismatch.'
    if BOOTSTRAP == 'offline':
        wheelhouse = Path(config['GRAPHTRUST_WHEELHOUSE'])
        env_lock = Path(config['GRAPHTRUST_ENV_LOCK'])
        assert wheelhouse.is_dir() and env_lock.is_file()
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--require-hashes', '--find-links', str(wheelhouse), '-r', str(env_lock)], check=True)
    else:
        assert importlib.util.find_spec('torch') is not None, "Use Kaggle's CUDA-compatible preinstalled torch."
        pins = ['transformers==5.19.0', 'Jinja2==3.1.6']
        if importlib.util.find_spec('accelerate') is None:
            pins.append('accelerate==1.10.1')
        subprocess.run([sys.executable, '-m', 'pip', 'install', *pins], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', str(wheel)], check=True)
    working.mkdir(parents=True, exist_ok=True)
    resolved = subprocess.run([sys.executable, '-m', 'pip', 'freeze', '--all'], check=True, capture_output=True, text=True).stdout
    (working / 'environment-resolved.txt').write_text(resolved)
    environment = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'accelerate', 'Jinja2', 'graphtrust-swe')}
    (working / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
    print(environment)
else:
    print('Package installation skipped: run_real is false.')


## Verify checkpoint provenance and prepared contexts

Hash the actual runtime checkpoint files, including weights and tokenizer/template assets. A mounted directory name does not establish immutable identity. Optional supplied checksums are verified; the complete runtime manifest is preserved. Context checks refuse fewer than ten qualified development issues, missing arms, approximate token counts, changed source, or evaluator markers in prompts.


In [ ]:
if RUN_REAL:
    from graphtrust.inference import checkpoint_manifest
    model_dir = Path(config['GRAPHTRUST_MODEL_DIR']).resolve()
    assert model_dir.is_dir(), 'Supply the actual Google Gemma 4 E2B model directory.'
    model_manifest = checkpoint_manifest(model_dir)
    (working / 'model-manifest-used.json').write_text(json.dumps(model_manifest, indent=2) + '\n')
    assert model_manifest and model_manifest['files']
    if config['GRAPHTRUST_MODEL_MANIFEST']:
        supplied = json.loads(Path(config['GRAPHTRUST_MODEL_MANIFEST']).read_text())
        assert supplied.get('model_id', MODEL_ID) == MODEL_ID
        for relative, expected in supplied['files'].items():
            artifact = (model_dir / relative).resolve()
            assert model_dir in artifact.parents, 'Model manifest path escapes checkpoint directory.'
            assert artifact.is_file() and sha256(artifact) == expected, 'Model artifact checksum mismatch: ' + relative
    context_dir = Path(config['GRAPHTRUST_CONTEXT_DIR'])
    contexts = [json.loads(line) for line in (context_dir / 'contexts.jsonl').read_text().splitlines() if line.strip()]
    assert len(contexts) == 60 and len({r['case_id'] for r in contexts}) == 10, 'Pilot requires ten qualified issues and all six conditions.'
    assert all(r['token_method'] == 'tokenizer' for r in contexts), 'Rebuild contexts with the actual model processor.'
    from graphtrust.prediction import RESPONSE_CONTRACT_VERSION
    assert all(r.get('response_contract_version') == RESPONSE_CONTRACT_VERSION for r in contexts), 'Rebuild fresh response-contract v2 contexts.'
    assert all(r['input_tokens'] <= 4000 for r in contexts)
    assert all(all(marker not in r['prompt'] for marker in ('known_false_edge_ids', 'test_patch', '\"patch\":')) for r in contexts)
    protocol_file = Path(config['GRAPHTRUST_PROTOCOL_JSON'])
    protocol = json.loads(protocol_file.read_text())
    manifest = json.loads((context_dir / 'manifest.json').read_text())
    assert protocol.get('schema_version') == manifest.get('schema_version') == 2
    assert protocol.get('response_contract_version') == manifest.get('response_contract_version') == RESPONSE_CONTRACT_VERSION
    base_command = [sys.executable, '-m', 'graphtrust', '--protocol', str(protocol_file)]
    subprocess.run(base_command + ['validate', '--directory', str(context_dir)], check=True)
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
else:
    print('Model/context validation skipped: run_real is false.')


## Execute the development pilot

Only enable this cell in a user-selected free T4 session. Generation is limited to 120 planned calls, two paired seeds, and the protocol's 256-token output cap with thinking disabled. Resume keeps terminal failures. The runner's measured timeout classification is documented; it is not a promise to forcibly stop GPU work at 90 seconds.


In [ ]:
if RUN_REAL:
    import shutil
    from graphtrust.inference import TransformersBackend, run_contexts
    protocol = json.loads(protocol_file.read_text())
    assert protocol['max_output_tokens'] <= 256
    import torch
    assert torch.cuda.is_available(), 'Select a free Kaggle T4 GPU session before enabling real inference.'
    gpu_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
    assert all('T4' in name for name in gpu_names), 'This primary pilot is configured for free T4 GPUs.'
    (working / 'hardware.json').write_text(json.dumps({'gpu_names': gpu_names, 'torch_cuda': torch.version.cuda}, indent=2) + '\n')
    backend = TransformersBackend(str(model_dir), local_files_only=True)
    (working / 'backend-identity.json').write_text(json.dumps(backend.identity, indent=2) + '\n')
    from graphtrust.io import digest
    from graphtrust.provenance import implementation_sha256
    context_manifest = json.loads((context_dir / 'manifest.json').read_text())
    assert context_manifest['implementation_sha256'] == implementation_sha256(), 'Wheel code differs from context preparation.'
    assert context_manifest['protocol_sha256'] == digest(protocol), 'Protocol identity changed.'
    assert context_manifest['tokenizer_identity'] == backend.identity['tokenizer_sha256'], 'Tokenizers differ: rebuild all contexts before inference.'
    for row in contexts:
        actual_tokens = backend.count_tokens(row['prompt'])
        assert actual_tokens == row['input_tokens'], 'Context tokenizer framing differs from inference: rebuild every arm.'
        assert actual_tokens <= protocol['max_input_tokens'] <= 4000
    working.mkdir(parents=True, exist_ok=True)
    for filename in ('contexts.jsonl', 'manifest.json'):
        shutil.copy2(context_dir / filename, working / filename)
    records_path = working / 'records.jsonl'
    run_summary = run_contexts(contexts, records_path, backend, SEEDS, MODEL_ID + '@sha256:' + model_manifest['sha256'], max_output_tokens=protocol['max_output_tokens'], timeout_seconds=protocol['timeout_seconds'], temperature=protocol['temperature'], max_calls=MAX_CALLS)
    print(json.dumps(run_summary, indent=2))
    if config['GRAPHTRUST_ENV_LOCK']:
        (working / 'environment-lock-used.sha256').write_text(sha256(config['GRAPHTRUST_ENV_LOCK']) + '\n')
else:
    print('Real inference skipped. No pilot result has been generated.')


## Review pilot gates without gold

The CLI freeze command is the single source of truth for acceptance. It checks complete real records, exact tokenizer manifests, parseability, memory/time behavior, and graph evidence use. A failure is saved with its diagnostic; it must not be presented as a frozen protocol. A low citation rate requires development-only presentation review, not a hidden instruction forcing graph reliance. The evaluator later scores localization against isolated gold.


In [ ]:
if RUN_REAL:
    freeze = subprocess.run(base_command + ['freeze', '--pilot', str(working), '--out', str(working / 'protocol-frozen.json')], capture_output=True, text=True)
    diagnostic = {
        'status': 'frozen' if freeze.returncode == 0 else 'pilot_gate_failed',
        'exit_code': freeze.returncode,
        'stdout': freeze.stdout,
        'stderr': freeze.stderr,
    }
    (working / 'pilot-gate-diagnostic.json').write_text(json.dumps(diagnostic, indent=2) + '\n')
    print(json.dumps(diagnostic, indent=2))
else:
    print('Pilot freeze is pending real records; no frozen protocol was written.')


## Preserve and analyze

Download `records.jsonl`, its companion attempt journal, model identity, environment lock hash, and gate diagnostics. On the evaluator machine, score with:

```text
python -m graphtrust report --records runs/pilot/records.jsonl --contexts runs/pilot/contexts.jsonl --gold data/swebench-lite/evaluator/gold.jsonl --out runs/pilot/report
python -m graphtrust freeze --pilot runs/pilot --out runs/protocol-frozen.json
```

A complete real pilot and exact-tokenizer source packs are prerequisites for freeze. Budget revisions apply uniformly to all conditions and require a new versioned pilot. Never combine runtime changes into one model stratum. The real holdout uses the accepted frozen protocol; this development notebook contains no holdout tuning.
